# Saving and exporting a YOLO model

### Key words

| Word | Meaning | Example |
|---|---|---|
| Model file | The trained model, stored on disk | `yolo26n.pt` |
| Export | Convert the model to another format | `.pt` → `.onnx` |
| PyTorch (`.pt`) | The original format. It can be trained further | `yolo26n.pt` |
| TorchScript | A PyTorch format that runs without the Ultralytics code | `yolo26n.torchscript` |
| ONNX | Open Neural Network Exchange: a common format for many systems | `yolo26n.onnx` |
| Runtime | The program that runs a model file | ONNX Runtime (from Microsoft) |
| Platform | The system where the model runs | Windows, Linux, a cloud server, a phone |
| CPU / GPU | Processor / graphics card. A GPU is much faster for models | `device="cpu"` or `device=0` |
| Container | A package with the application, its libraries and the model | A Docker image on Google Cloud Run |
| Cloud Storage bucket | A folder in the cloud for files | `gs://my-bucket/models/` |

## 0. Setup
Run this cell once. Expected output: `True`.

In [1]:
import sys
print("Using this folder's uv environment:", ".venv" in sys.executable)

import os
import pandas as pd
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
os.makedirs("outputs", exist_ok=True)

Using this folder's uv environment: True


## 1. Three formats

| Format | File | Runs with | Use it to |
|---|---|---|---|
| PyTorch | `.pt` | Python, PyTorch and Ultralytics | Train further and experiment |
| TorchScript | `.torchscript` | PyTorch, from Python or C++ | Run in PyTorch-based applications |
| ONNX | `.onnx` | ONNX Runtime: Windows, Linux, macOS; Python, C++, C#, Java, JavaScript | **Share across systems and platforms** |

**Rule of thumb:** keep the `.pt` file to train further. Share the `.onnx` file to use the model elsewhere.

**Security:** loading a `.pt` file can run code, so load only files from trusted sources. An `.onnx` file contains no Python code.

## 2. Save a model
`save()` writes a `.pt` copy. This copy keeps everything needed to train further.

In [2]:
model.save("outputs/my_model.pt")
print("Saved: outputs/my_model.pt")

Saved: outputs/my_model.pt


## 3. Export to TorchScript and ONNX
Each export creates a new file next to `yolo26n.pt`.
`simplify=False` skips an optional clean-up step that needs an extra library.

In [3]:
torchscript_file = model.export(format="torchscript")
onnx_file = model.export(format="onnx", simplify=False)

print("TorchScript:", torchscript_file)
print("ONNX:", onnx_file)

Ultralytics 8.4.184  Python-3.12.13 torch-2.14.1+cpu CPU (13th Gen Intel Core i9-13900HX)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

TorchScript: starting export with torch 2.14.1+cpu...


c:\Users\demo5\Documents\Apps\Case-YOLO\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
c:\Users\demo5\Documents\Apps\Case-YOLO\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


TorchScript: export success  0.8s, saved as 'yolo26n.torchscript' (9.8 MB)

Export complete (1.1s)
Results saved to C:\Users\demo5\Documents\Apps\Case-YOLO\yolo26n.torchscript
Predict:         yolo predict task=detect model=yolo26n.torchscript imgsz=640 
Validate:        yolo val task=detect model=yolo26n.torchscript imgsz=640  
Visualize:       https://netron.app
Ultralytics 8.4.184  Python-3.12.13 torch-2.14.1+cpu CPU (13th Gen Intel Core i9-13900HX)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: export success  0.8s, saved as 'yolo26n.onnx' (9.4 MB)

Export complete (0.9s)
Results saved to C:\Users\demo5\Documents\Apps\Case-YOLO\yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz

## 4. Load each file and compare the results
Ultralytics opens every format with the same line: `YOLO(file)`.
The objects found should be the same. Small differences in confidence are normal.

In [4]:
rows = []
for file in ["yolo26n.pt", torchscript_file, onnx_file]:
    r = YOLO(file)("example1.jpeg", verbose=False)[0]
    found = ", ".join(f"{r.names[int(c)]} {p:.2f}" for c, p in zip(r.boxes.cls, r.boxes.conf))
    rows.append({"file": file, "size (MB)": round(os.path.getsize(file) / 1e6, 1), "objects found": found})

pd.DataFrame(rows)

Loading yolo26n.torchscript for TorchScript inference...


c:\Users\demo5\Documents\Apps\Case-YOLO\.venv\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.31.0 with CPUExecutionProvider


,file,size (MB),objects found
0,yolo26n.pt,5.5,"cat 0.75, bed 0.29, couch 0.28"
1,yolo26n.torchscript,10.3,"cat 0.73, bed 0.48"
2,yolo26n.onnx,9.9,"cat 0.73, bed 0.48"


## 5. Inside an ONNX file
ONNX Runtime (from Microsoft) opens the file without Ultralytics or PyTorch.
The model is a fixed calculation: numbers in, numbers out.

In [5]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_file)
print("Input: ", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

Input:  [1, 3, 640, 640]
Output: [1, 84, 8400]


**What to observe:** the input `[1, 3, 640, 640]` means 1 image, 3 colours, 640 × 640 pixels.
The output holds the found objects: for each one, the box corners, the confidence and the class.

## 6. Where to store the files

| Place | When to use it |
|---|---|
| The project folder | Learning and tests |
| Cloud storage, e.g. a Google Cloud Storage bucket | Several machines download the same file |
| Inside a container image | The model ships with the application, e.g. on Google Cloud Run |

**Good practice**
1. Put a version in the file name. Example: `vehicles_v1.onnx`.
2. Record the file's fingerprint (SHA-256), to check later that it has not changed.
3. Use the same library versions everywhere. In this project they are saved in `uv.lock`.
4. Before use on a new system, compare its results with the original, as in section 4.

## 7. A note on GPUs

The same model files run on a CPU or a GPU. Only one setting changes:

| Setting | Meaning |
|---|---|
| `device="cpu"` | Use the processor |
| `device=0` | Use the first GPU |
| `device=[0, 1]` | Train on two GPUs |

Example: `model("example1.jpeg", device=0)`.
For ONNX files on a GPU, the GPU version of ONNX Runtime is used (`onnxruntime-gpu`).